In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<table align="left">
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Fgooglemaps-samples%2Finsights-samples%2Fmain%2Fstreet_view_insights%2Fnotebooks%2FSupervised%20fine%20tuning%2FSupervised_fine_tuned_Gemini_based_lamp_identificationv2.ipynb?utm_source=cropped_street_view_insights_notebooks">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
</table>

# Detect if a lamp is an acquity brand lamp using Supervised fine tuned Gemini

## Overview

This codelab walks you through the process of building a simple image classification system using a pre-trained and then fine-tuned Gemini model. The goal is to accurately identify whether a given lamp image belongs to the "Acquity" brand.

The process involves:

1.  **Loading the pre-trained Gemini model:** We start by initializing the generative AI model.
2.  **Preparing the dataset:** You'll need a dataset of lamp images, labeled as either "Acquity" or "Not Acquity". This dataset will be used for fine-tuning.
3.  **Fine-tuning the model:** The codelab will guide you through the steps of fine-tuning the Gemini model on your specific dataset. This process adapts the model's knowledge to better recognize Acquity lamps.
4.  **Evaluating the model:** After fine-tuning, we'll test the model's performance on unseen data to assess its accuracy.
5.  **Making predictions:** Finally, you'll learn how to use the fine-tuned model to predict whether a new lamp image is an Acquity brand lamp.

By the end of this codelab, you will have a working example of how to leverage the power of large language models for specific image classification tasks through supervised fine-tuning.


In [ ]:
# Install the necessary library
!pip install --upgrade --user --quiet google-genai google-cloud-aiplatform

## Setup
Enable APIs and Set Permissions
Enable the Vertex AI API

Make sure you have been granted the roles for the GCP project you'll access from this notebook:

roles/aiplatform.user

## Configuration

**Important**: Replace the placeholder values below with your actual GCP Project ID and Region.

In [ ]:
PROJECT_ID = ''  # @param {type:"string"}
REGION = ''      # @param {type:"string"}

# BigQuery Configuration
BIGQUERY_DATASET_ID = '' # @param {type:"string"}
BIGQUERY_TABLE_ID = 'cropped_observations_latest' # @param {type:"string"}
QUERY_LIMIT = 10 # @param {type:"integer"}
ASSET_TYPE = "ASSET_CLASS_UTILITY_POLE" # @param {type:"string"}
MODEL = "gemini-3.5-flash" # @param {type:"string"}


### ⚠️ IAM Permissions Required for Vertex AI Tuning

Before proceeding, ensure that the Vertex AI tuning service agent has read access to your Cloud Storage bucket. 

You must grant the **Storage Object Viewer** (`roles/storage.objectViewer`) IAM role on your bucket to the following service account:
`service-<YOUR_PROJECT_NUMBER>@gcp-sa-vertex-tune.iam.gserviceaccount.com`

*(Note: Replace `<YOUR_PROJECT_NUMBER>` with your actual Google Cloud Project Number, which you can find in the Google Cloud Console dashboard).*


In [ ]:
# Provide a bucket name
BUCKET_NAME = ""  # @param {type:"string"}
BUCKET_URI = f"gs://{BUCKET_NAME}"
LOCATION = "us"

# Create the bucket if it doesn't exist
!gcloud storage ls --buckets {BUCKET_URI} || gcloud storage buckets create --location {LOCATION} --project {PROJECT_ID} {BUCKET_URI}

In [ ]:
# Upload the training data to your bucket
import os

GITHUB_REPOSITORY_NAME = 'googlemaps-samples/insights-samples' # @param {type:"string"}
GITHUB_BRANCH_NAME = 'main' # @param {type:"string"}

# 1. Define the base path and the training file name
BASE_URL = f"https://raw.githubusercontent.com/{GITHUB_REPOSITORY_NAME}/{GITHUB_BRANCH_NAME}/street_view_insights/cropped/notebooks/Supervised%20fine%20tuning/data/"

TRAINING_FILE_NAME = "acquity_detector_negative_examples_v2.jsonl"

# 2. Construct the RAW GitHub URL
RAW_URL = f"{BASE_URL}{TRAINING_FILE_NAME}"

# 3. Download the file directly to the Colab VM using wget
!wget "$RAW_URL" -O "$TRAINING_FILE_NAME"

# 4. Set environment variables to pass to the gcloud command
os.environ["TRAINING_FILE_NAME"] = TRAINING_FILE_NAME
os.environ["BUCKET_URI"] = BUCKET_URI

# 5. Upload the downloaded file to your GCS bucket
!gcloud storage cp "$TRAINING_FILE_NAME" "$BUCKET_URI/data/"


In [ ]:
# Clean training data by verifying HTTPS links and upload the cleaned jsonl to GCS bucket

import json
import os
import requests
from google.cloud import storage

FILTERED_FILE_NAME = "acquity_detector_negative_examples_v2_clean.jsonl"

print("Starting to verify HTTP links in the dataset...")

# Initialize the Cloud Storage client and get the target bucket
storage_client = storage.Client(project=PROJECT_ID)
bucket = storage_client.bucket(BUCKET_NAME)

clean_data = []
total_processed = 0
total_discarded = 0

# Process the original dataset file line by line
with open(TRAINING_FILE_NAME, 'r') as f:
    for line_number, line in enumerate(f):
        total_processed += 1
        example = json.loads(line)
        keep_example = True

        # Traverse the JSON structure to find image URIs
        for content in example.get("contents", []):
            for part in content.get("parts", []):
                file_data = part.get("fileData", {})

                # Identify URLs that attempt to fetch images directly from the internet
                if "fileUri" in file_data and file_data["fileUri"].startswith("https://"):
                    img_url = file_data["fileUri"]

                    try:
                        # Use stream=True to quickly check the status code without downloading the image body
                        headers = {'User-Agent': 'Mozilla/5.0'}
                        img_response = requests.get(img_url, headers=headers, stream=True, timeout=10)

                        # If the server returns a 200 OK, we do nothing and keep the https:// link intact!
                        if img_response.status_code == 200:
                            print(f"[{line_number}] Kept: HTTP 200 OK for {img_url}")
                        else:
                            print(f"[{line_number}] Discarded: HTTP {img_response.status_code} for {img_url}")
                            keep_example = False  # Link is broken or blocked, flag for removal

                    except Exception as e:
                        print(f"[{line_number}] Discarded: Error fetching URL: {e}")
                        keep_example = False  # Request error, flag for removal

        # Only retain examples that have a valid, accessible image
        if keep_example:
            clean_data.append(example)
        else:
            total_discarded += 1

# Save the newly filtered JSONL file locally
with open(FILTERED_FILE_NAME, 'w') as f:
    for item in clean_data:
        f.write(json.dumps(item) + '\n')

# Upload the final, clean JSONL file to Cloud Storage
dataset_blob = bucket.blob(f"data/{FILTERED_FILE_NAME}")
dataset_blob.upload_from_filename(FILTERED_FILE_NAME)

new_dataset_uri = f"{BUCKET_URI}/data/{FILTERED_FILE_NAME}"
print(f"\nSuccess! Your clean dataset is uploaded to: {new_dataset_uri}")
print(f"Processed: {total_processed} total examples.")
print(f"Retained:  {len(clean_data)} valid examples for fine-tuning.")
print(f"Removed:   {total_discarded} invalid examples.")


In [ ]:
# Import the library
import google.cloud.aiplatform as aiplatform
from google import genai
from google.genai import types
import time

# Initialize the Vertex AI SDK and Gen AI Client
aiplatform.init(project=PROJECT_ID, location=REGION)
client = genai.Client(vertexai=True, project=PROJECT_ID, location=REGION)

# Define the dataset URI and model name
dataset_uri = new_dataset_uri
tuned_model_display_name = "Aquity_model_detector_fine_tuned"
base_model = MODEL

training_dataset = {
    "gcs_uri": dataset_uri,
}

# Tune a model using `tune` method.
sft_tuning_job = client.tunings.tune(
    base_model=base_model,
    training_dataset=training_dataset,
    config=types.CreateTuningJobConfig(
        tuned_model_display_name=tuned_model_display_name,
    ),
)

# Get the tuning job info.
tuning_job = client.tunings.get(name=sft_tuning_job.name)

# Status Check
print("Tuning job created. Waiting for completion...")
# Wait for job completion
running_states = [
    "JOB_STATE_PENDING",
    "JOB_STATE_RUNNING",
]

while tuning_job.state.name in running_states:
    print(".", end="")
    tuning_job = client.tunings.get(name=tuning_job.name)
    time.sleep(60) # Check every minute

print()

if tuning_job.state.name == "JOB_STATE_SUCCEEDED":
    MODEL_ENDPOINT = tuning_job.tuned_model.endpoint
    print(f"Model deployed to endpoint: {MODEL_ENDPOINT}")
else:
    error_msg = f"Tuning job failed with state: {tuning_job.state.name}"
    if hasattr(tuning_job, 'error') and tuning_job.error:
        error_msg += f" - Error: {tuning_job.error}"
    raise Exception(error_msg)


In [ ]:
from google.cloud import bigquery
from google.genai.types import Part
import pandas as pd

In [ ]:
# Updated query to group by asset_id and aggregate all image URIs
BIGQUERY_QUERY = f"""
    SELECT
        asset_id,
        ARRAY_AGG(gcs_uri) AS gcs_uris
    FROM
        `{PROJECT_ID}.{BIGQUERY_DATASET_ID}.{BIGQUERY_TABLE_ID}`
    WHERE
        asset_type = '{ASSET_TYPE}'
        AND gcs_uri IS NOT NULL
    GROUP BY
        asset_id
    LIMIT {QUERY_LIMIT}
"""

In [ ]:
# Updated prompt to be more lenient and ask for model number
PROMPT = '''Follow these rules precisely to generate your answer:

1.  **Analyze the Input:** Carefully examine the provided Lamp Input. Compare its specific features, design elements, markings, and any visible model numbers against the information in the Acuity Brand Identification Guide.

2.  **Generate a Confidence Score:** Based on your analysis, internally generate a confidence score from 0.0 to 1.0 that represents your certainty that the lamp is an Acuity brand product.

3.  **Apply Strict Classification Logic:** Use your confidence score to determine your answer according to the following thresholds:
    *   **If the score is greater than 0.5:** Your answer is "Yes". This indicates a high degree of confidence, requiring a direct match of multiple key features, design language, or a model number consistent with the guide.
    *   **If the score is between 0.25 and 0.5 (inclusive):** Your answer is "Maybe". This indicates some features align with the guide, but there is not enough evidence for a confident "Yes" or "No".
    *   **If the score is less than 0.25:** Your answer is "No". This indicates the lamp has features that contradict the guide, is identifiable as a different brand, or lacks any resemblance to an Acuity product.

4.  **Format Your Output:** Your response must strictly follow one of the formats below, based on the answer you determined in the previous step. Do not add any extra text, explanations, or apologies.
    *   **For a "Yes" answer:**
        *   If a model number is visible or mentioned in the Lamp Input, respond with: `Yes. Model Number: [model number]`
        *   If no model number is available, respond with: `Yes.`

    *   **For a "Maybe" answer:**
        *   Respond with: `Maybe.`

    *   **For a "No" answer:**
        *   Respond with: `No.`'''

In [ ]:
NUM_ROWS_TO_PROCESS = 10

In [ ]:
# ------------ 1. Fetch data from BigQuery ------------
# Create a BigQuery client
client = bigquery.Client(project=PROJECT_ID)

# Execute the query and load the results into a pandas DataFrame
df = client.query(BIGQUERY_QUERY).to_dataframe()

In [ ]:
# ------------ 2. Initialize Gen AI Client and Load Model ------------
from google import genai
from google.genai import types

# Parse project and location from MODEL_ENDPOINT to ensure they match
endpoint_parts = MODEL_ENDPOINT.split('/')
parsed_project = endpoint_parts[1]
parsed_region = endpoint_parts[3]

# Initialize the new Gen AI Client
client = genai.Client(vertexai=True, project=parsed_project, location=parsed_region)

In [ ]:
# ------------ 3. Analyze Images ------------
# Create an empty list to store the analysis results
analysis_results = []

# Create a subset of the DataFrame to process
df_to_process = df.head(NUM_ROWS_TO_PROCESS)
for index, row in df_to_process.iterrows():
    # Get the list of GCS URIs for the asset
    image_uris = row['gcs_uris']
    asset_id = row['asset_id']
    print(f"Processing asset: {asset_id}")

    if image_uris is not None and len(image_uris) > 0:
        # Prepare the content for the model: prompt + all images
        content_parts = [PROMPT]
        for uri in image_uris:
            if uri: # Ensure URI is not None
                # Use the new SDK's Part syntax
                content_parts.append(
                    types.Part(file_data={'file_uri': uri, 'mime_type': 'image/jpeg'})
                )

        # Generate content using the new client
        response = client.models.generate_content(
            model=MODEL_ENDPOINT,
            contents=content_parts
        )
        result_text = response.text.strip()
        print(f"  -> Result: {result_text}")
        analysis_results.append(result_text)

    else:
        no_uri_message = "No GCS URIs found for this asset."
        print(f"  -> {no_uri_message}")
        analysis_results.append(no_uri_message)


In [ ]:
# ------------ 4. Display All Results ------------
# Create a new DataFrame with only asset_id and analysis_result
results_df = pd.DataFrame({
    'asset_id': df_to_process['asset_id'],
    'analysis_result': analysis_results
})

# Display the full DataFrame without filtering
print("\n--- Final Results --- Succeeded")
display(results_df)

In [ ]:
# dataframe: results_df
# uuid: DD0E9F14-06D2-45C9-98A8-12395A8FB640
# output_variable:
# config_str: CpYMeyJjaGFydENvbmZpZyI6eyJkYXRhc291cmNlSWQiOiJfX1ZJWl9EQVRBU09VUkNFX18iLCJwcm9wZXJ0eUNvbmZpZyI6eyJjb21wb25lbnRQcm9wZXJ0eSI6eyJzb3J0IjpbeyJzb3J0RGlyIjoxLCJzb3J0Q29sdW1uIjoicXRfZWl6ZDdhcTg1ZCJ9XSwiYnJlYWtkb3duQ29uZmlnIjpbXSwiZmlsdGVycyI6W10sImluaGVyaXRGaWx0ZXJzIjp0cnVlLCJkc1JlcXVpcmVkRmlsdGVycyI6W10sImRhdGFzZXQiOnsiZGF0YXNldFR5cGUiOjEsImRhdGFzZXRJZCI6Il9fVklaX0RBVEFTT1VSQ0VfXyJ9LCJyb3ciOjEwMCwiZGltZW5zaW9ucyI6eyJsYWJlbGVkQ29uY2VwdHMiOlt7ImtleSI6InByaW1hcnkiLCJ2YWx1ZSI6eyJjb25jZXB0TmFtZXMiOlsicXRfMDEwZDdhcTg1ZCIsInF0XzExMGQ3YXE4NWQiXX19XX0sIm1ldHJpY3MiOnsibGFiZWxlZENvbmNlcHRzIjpbeyJrZXkiOiJwcmltYXJ5IiwidmFsdWUiOnsiY29uY2VwdE5hbWVzIjpbXX19XX0sInRhYmxlUHJvcGVydHkiOnsiY29sdW1uc1dpZHRoIjpbNC4xODY0MDM0MTQ1ODY1ODEsNDEzLjQwNjc5ODI5MjcwNjcsNDEzLjQwNjc5ODI5MjcwNjddLCJyb3dzSGVpZ2h0IjpbMzEsMzEsMzFdLCJ0YWJsZURpbWVuc2lvblByb3BlcnR5IjpbXSwidGFibGVNZXRyaWNQcm9wZXJ0eSI6W10sImJhY2tncm91bmRBbmRCb3JkZXJQcm9wZXJ0eSI6eyJib3JkZXIiOnsib3BhY2l0eSI6MCwic2l6ZSI6MCwicmFkaXVzIjowfX19LCJjb21wb25lbnRQcm9wZXJ0eU1pZ3JhdGlvblN0YXR1cyI6Mn19LCJjb25jZXB0RGVmcyI6W3siaWQiOiJ0MC5xdF9laXpkN2FxODVkIiwibmFtZSI6InF0X2VpemQ3YXE4NWQiLCJuYW1lc3BhY2UiOiJ0MCIsInF1ZXJ5VGltZVRyYW5zZm9ybWF0aW9uIjp7ImRhdGFUcmFuc2Zvcm1hdGlvbiI6eyJzb3VyY2VGaWVsZE5hbWUiOiJhbmFseXNpc19yZXN1bHQiLCJhZ2dyZWdhdGlvbiI6M319fSx7ImlkIjoidDAucXRfMDEwZDdhcTg1ZCIsIm5hbWUiOiJxdF8wMTBkN2FxODVkIiwibmFtZXNwYWNlIjoidDAiLCJxdWVyeVRpbWVUcmFuc2Zvcm1hdGlvbiI6eyJkYXRhVHJhbnNmb3JtYXRpb24iOnsic291cmNlRmllbGROYW1lIjoiYXNzZXRfaWQifX19LHsiaWQiOiJ0MC5xdF8xMTBkN2FxODVkIiwibmFtZSI6InF0XzExMGQ3YXE4NWQiLCJuYW1lc3BhY2UiOiJ0MCIsInF1ZXJ5VGltZVRyYW5zZm9ybWF0aW9uIjp7ImRhdGFUcmFuc2Zvcm1hdGlvbiI6eyJzb3VyY2VGaWVsZE5hbWUiOiJhbmFseXNpc19yZXN1bHQifX19XSwiYXR0cmlidXRlQ29uZmlnIjp7ImNvbXBvbmVudEF0dHJpYnV0ZSI6eyJkYXRhc291cmNlQ29uZmlnVmVyc2lvbiI6MiwiZGlzcGxheUNvbmZpZ1ZlcnNpb24iOjAsImhlaWdodCI6NTkwLCJ3aWR0aCI6ODMxLCJ0b3AiOjAsImxlZnQiOjAsInJvdGF0aW9uIjowfX0sImNvbXBvbmVudElkIjoiX19WSVpfQ0hBUlRfSURfXyIsInR5cGUiOiJzaW1wbGUtdGFibGUiLCJwcmVzZXQiOiJkZWZhdWx0IiwiYmVoYXZpb3IiOnsibWFwVmFsdWUiOnsiZW50cnkiOltdfX19LCJmaWx0ZXJzIjpbXSwiY2hhcnRJbnRlcmFjdGlvbnMiOltdLCJ2ZXJzaW9uIjoxfRoTCg9hbmFseXNpc19yZXN1bHQQARoMCghhc3NldF9pZBABIJ0F

import google.colabsqlviz.explore_dataframe as _vizcell
_vizcell.explore_dataframe(df_or_df_name='results_df', uuid='DD0E9F14-06D2-45C9-98A8-12395A8FB640', config_str='CpYMeyJjaGFydENvbmZpZyI6eyJkYXRhc291cmNlSWQiOiJfX1ZJWl9EQVRBU09VUkNFX18iLCJwcm9wZXJ0eUNvbmZpZyI6eyJjb21wb25lbnRQcm9wZXJ0eSI6eyJzb3J0IjpbeyJzb3J0RGlyIjoxLCJzb3J0Q29sdW1uIjoicXRfZWl6ZDdhcTg1ZCJ9XSwiYnJlYWtkb3duQ29uZmlnIjpbXSwiZmlsdGVycyI6W10sImluaGVyaXRGaWx0ZXJzIjp0cnVlLCJkc1JlcXVpcmVkRmlsdGVycyI6W10sImRhdGFzZXQiOnsiZGF0YXNldFR5cGUiOjEsImRhdGFzZXRJZCI6Il9fVklaX0RBVEFTT1VSQ0VfXyJ9LCJyb3ciOjEwMCwiZGltZW5zaW9ucyI6eyJsYWJlbGVkQ29uY2VwdHMiOlt7ImtleSI6InByaW1hcnkiLCJ2YWx1ZSI6eyJjb25jZXB0TmFtZXMiOlsicXRfMDEwZDdhcTg1ZCIsInF0XzExMGQ3YXE4NWQiXX19XX0sIm1ldHJpY3MiOnsibGFiZWxlZENvbmNlcHRzIjpbeyJrZXkiOiJwcmltYXJ5IiwidmFsdWUiOnsiY29uY2VwdE5hbWVzIjpbXX19XX0sInRhYmxlUHJvcGVydHkiOnsiY29sdW1uc1dpZHRoIjpbNC4xODY0MDM0MTQ1ODY1ODEsNDEzLjQwNjc5ODI5MjcwNjcsNDEzLjQwNjc5ODI5MjcwNjddLCJyb3dzSGVpZ2h0IjpbMzEsMzEsMzFdLCJ0YWJsZURpbWVuc2lvblByb3BlcnR5IjpbXSwidGFibGVNZXRyaWNQcm9wZXJ0eSI6W10sImJhY2tncm91bmRBbmRCb3JkZXJQcm9wZXJ0eSI6eyJib3JkZXIiOnsib3BhY2l0eSI6MCwic2l6ZSI6MCwicmFkaXVzIjowfX19LCJjb21wb25lbnRQcm9wZXJ0eU1pZ3JhdGlvblN0YXR1cyI6Mn19LCJjb25jZXB0RGVmcyI6W3siaWQiOiJ0MC5xdF9laXpkN2FxODVkIiwibmFtZSI6InF0X2VpemQ3YXE4NWQiLCJuYW1lc3BhY2UiOiJ0MCIsInF1ZXJ5VGltZVRyYW5zZm9ybWF0aW9uIjp7ImRhdGFUcmFuc2Zvcm1hdGlvbiI6eyJzb3VyY2VGaWVsZE5hbWUiOiJhbmFseXNpc19yZXN1bHQiLCJhZ2dyZWdhdGlvbiI6M319fSx7ImlkIjoidDAucXRfMDEwZDdhcTg1ZCIsIm5hbWUiOiJxdF8wMTBkN2FxODVkIiwibmFtZXNwYWNlIjoidDAiLCJxdWVyeVRpbWVUcmFuc2Zvcm1hdGlvbiI6eyJkYXRhVHJhbnNmb3JtYXRpb24iOnsic291cmNlRmllbGROYW1lIjoiYXNzZXRfaWQifX19LHsiaWQiOiJ0MC5xdF8xMTBkN2FxODVkIiwibmFtZSI6InF0XzExMGQ3YXE4NWQiLCJuYW1lc3BhY2UiOiJ0MCIsInF1ZXJ5VGltZVRyYW5zZm9ybWF0aW9uIjp7ImRhdGFUcmFuc2Zvcm1hdGlvbiI6eyJzb3VyY2VGaWVsZE5hbWUiOiJhbmFseXNpc19yZXN1bHQifX19XSwiYXR0cmlidXRlQ29uZmlnIjp7ImNvbXBvbmVudEF0dHJpYnV0ZSI6eyJkYXRhc291cmNlQ29uZmlnVmVyc2lvbiI6MiwiZGlzcGxheUNvbmZpZ1ZlcnNpb24iOjAsImhlaWdodCI6NTkwLCJ3aWR0aCI6ODMxLCJ0b3AiOjAsImxlZnQiOjAsInJvdGF0aW9uIjowfX0sImNvbXBvbmVudElkIjoiX19WSVpfQ0hBUlRfSURfXyIsInR5cGUiOiJzaW1wbGUtdGFibGUiLCJwcmVzZXQiOiJkZWZhdWx0IiwiYmVoYXZpb3IiOnsibWFwVmFsdWUiOnsiZW50cnkiOltdfX19LCJmaWx0ZXJzIjpbXSwiY2hhcnRJbnRlcmFjdGlvbnMiOltdLCJ2ZXJzaW9uIjoxfRoTCg9hbmFseXNpc19yZXN1bHQQARoMCghhc3NldF9pZBABIJ0F')

## Generate Summary of Analysis Results


In [ ]:
try:
    def categorize_result(result):
        if not isinstance(result, str):
            return 'Error'
        if result.startswith('Yes'):
            return 'Yes'
        elif result == 'Maybe.':
            return 'Maybe'
        elif result == 'No.':
            return 'No'
        else:
            return 'Error'

    results_df['category'] = results_df['analysis_result'].apply(categorize_result)

    summary_df = results_df.groupby('category').agg(
        count=('asset_id', 'size'),
        sample_assets=('asset_id', lambda x: list(x.head(3)))
    ).reset_index()

    print("\n--- Analysis Summary ---")
    display(summary_df)

except NameError:
    print("Error: The 'results_df' DataFrame is not defined. Please ensure the previous cell has been executed successfully.")